In [ ]:
import os
import re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from collections import Counter
import argparse
import random
from sklearn.metrics import classification_report


## FastText Embedding

In [ ]:
import fasttext
import fasttext.util
from pyidaungsu import tokenize

# Load pretrained Burmese model (download first)
# https://fasttext.cc/docs/en/crawl-vectors.html
fasttext.util.download_model('my', if_exists='ignore')  # Burmese
ft_model = fasttext.load_model("cc.my.300.bin")

In [ ]:
!pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 2.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.2-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.2-py3-none-any.whl (310 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4647420 sha256=2d57f6511b69ee7a6d95af34cfacc4ff7c2431474ce74acc4510a6bb5fffda65
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [ ]:
class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, ft_model, max_len=50):
        self.texts = texts
        self.labels = labels
        self.ft_model = ft_model
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def encode_text(self, text):
        # ✅ Burmese word segmentation
        tokens = tokenize(str(text), form="word")
        print("Tokens are", tokens)

        # ✅ fallback if tokenizer fails
        if len(tokens) == 0:
            tokens = list(str(text))

        vectors = []
        for tok in tokens[:self.max_len]:
            vec = self.ft_model.get_word_vector(tok)
            vectors.append(vec)

        # ✅ padding
        while len(vectors) < self.max_len:
            vectors.append([0.0] * 300)

        return torch.tensor(vectors, dtype=torch.float)

    def __getitem__(self, idx):
        x = self.encode_text(self.texts[idx])
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

In [ ]:
import torch.nn as nn

class EmotionalBiLSTM(nn.Module):
    def __init__(self, embedding_dim=300, hidden_dim=128, output_dim=6):
        super().__init__()
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        # x: (batch, seq_len, 300)
        _, (hidden, _) = self.lstm(x)

        # Concatenate forward + backward
        h = torch.cat((hidden[-2], hidden[-1]), dim=1)

        return self.fc(self.dropout(h))

In [ ]:
import torch
import pandas as pd
from torch.utils.data import DataLoader, random_split
import torch.optim as optim

class HybridEliza:
    def __init__(self, ft_model, lang="my", model_path="eliza_burmese_ft.pth"):
        self.lang = lang
        #self.script = SCRIPTS[lang]
        #self.script["keywords"].sort(key=lambda x: x[2], reverse=True)

        self.ft_model = ft_model
        self.model_path = model_path
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.id2label = {
            0: "Sadness", 1: "Joy", 2: "Love",
            3: "Anger", 4: "Fear", 5: "Surprise"
        }

        self.model = None

    def train(self, data_path, epochs=5, lr=1e-3, batch_size=32, val_split=0.1):
        df = pd.read_csv(data_path)
         # ✅ SHUFFLE DATASET FIRST
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)

        label_col = 'Label ' if 'Label ' in df.columns else 'emotions'

        dataset = EmotionDataset(
            df['Text '].tolist(),
            df[label_col].tolist(),
            self.ft_model
        )

        val_size = int(len(dataset) * val_split)
        train_size = len(dataset) - val_size

        train_ds, val_ds = random_split(dataset, [train_size, val_size])

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size)

        self.model = EmotionalBiLSTM().to(self.device)

        optimizer = optim.Adam(self.model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()

        print(f"[*] Training on {self.device} (fastText Burmese)...")

        for epoch in range(epochs):
            self.model.train()
            total_loss = 0

            for batch_x, batch_y in train_loader:
                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)

                optimizer.zero_grad()
                outputs = self.model(batch_x)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            val_acc = self.evaluate(val_loader)

            print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2%}")

        torch.save(self.model.state_dict(), self.model_path)

    def evaluate(self, loader):
        self.model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(self.device), y.to(self.device)
                outputs = self.model(x)
                preds = torch.argmax(outputs, dim=1)

                correct += (preds == y).sum().item()
                total += y.size(0)

        return correct / total

    def predict(self, text):
        self.model.eval()
        dataset = EmotionDataset([text], [0], self.ft_model)
        x, _ = dataset[0]
        x = x.unsqueeze(0).to(self.device)

        with torch.no_grad():
            output = self.model(x)
            pred = torch.argmax(output, dim=1).item()

        return self.id2label[pred]

In [ ]:
# Initialize
eliza = HybridEliza(ft_model)

# Train
eliza.train("/content/drive/MyDrive/burmese_emotion.csv", epochs=20)

# Predict
#print(eliza.predict("ငါအရမ်းဝမ်းသာတယ်"))

[*] Training on cuda (fastText Burmese)...
Epoch 1/20 | Loss: 1.7875 | Val Acc: 16.67%
Epoch 2/20 | Loss: 1.7441 | Val Acc: 31.67%
Epoch 3/20 | Loss: 1.6565 | Val Acc: 40.83%
Epoch 4/20 | Loss: 1.4895 | Val Acc: 48.33%
Epoch 5/20 | Loss: 1.3400 | Val Acc: 50.00%
Epoch 6/20 | Loss: 1.2142 | Val Acc: 59.17%
Epoch 7/20 | Loss: 1.1615 | Val Acc: 55.83%
Epoch 8/20 | Loss: 1.0810 | Val Acc: 60.83%
Epoch 9/20 | Loss: 1.1640 | Val Acc: 63.33%
Epoch 10/20 | Loss: 1.0493 | Val Acc: 67.50%
Epoch 11/20 | Loss: 0.9259 | Val Acc: 60.00%
Epoch 12/20 | Loss: 0.9188 | Val Acc: 65.00%
Epoch 13/20 | Loss: 0.8550 | Val Acc: 55.83%
Epoch 14/20 | Loss: 0.7960 | Val Acc: 65.00%
Epoch 15/20 | Loss: 0.8116 | Val Acc: 65.83%
Epoch 16/20 | Loss: 0.7845 | Val Acc: 62.50%
Epoch 17/20 | Loss: 0.6868 | Val Acc: 64.17%
Epoch 18/20 | Loss: 0.6431 | Val Acc: 63.33%
Epoch 19/20 | Loss: 0.6041 | Val Acc: 64.17%
Epoch 20/20 | Loss: 0.6396 | Val Acc: 65.00%


In [ ]:
# Predict
print(eliza.predict("ငါအရမ်းဝမ်းသာတယ်"))

Sadness


## Syllbreak

In [ ]:
!wget https://raw.githubusercontent.com/ye-kyaw-thu/sylbreak/master/python/sylbreak.py

--2026-03-19 13:40:26--  https://raw.githubusercontent.com/ye-kyaw-thu/sylbreak/master/python/sylbreak.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3668 (3.6K) [text/plain]
Saving to: ‘sylbreak.py’

sylbreak.py         100%[===================>]   3.58K  --.-KB/s    in 0s      

2026-03-19 13:40:26 (70.6 MB/s) - ‘sylbreak.py’ saved [3668/3668]



In [ ]:
from sylbreak import break_syllables, create_break_pattern

def syllable_break(text):
  """Syllable break for burmese texts"""
  text = text
  separator = ' '
  break_pattern = create_break_pattern()

  segmented = break_syllables(text, break_pattern, separator)
  return segmented

def myanmar_syllable_tokenize(text: str) -> list[str]:
    """Tokenize Myanmar text into syllables.

    Example:
        "ကျွန်တော်ကောင်းသည်" → ["ကျွန်", "တော်", "ကောင်း", "သည်"]
    """
    return syllable_break(text).split()

def normalize_myanmar(text: str) -> str:
    """Strip whitespace and collapse multiple spaces."""
    return re.sub(r"\s+", " ", text.strip())

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, word2id, max_len=100, lang="my"):
        self.texts = texts
        self.labels = labels
        self.word2id = word2id
        self.max_len = max_len
        self.lang = lang

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = normalize_myanmar(str(self.texts[idx]))
        if self.lang == "my":
            tokens = myanmar_syllable_tokenize(text)
        else:
            tokens = text.lower().split()
        seq = [self.word2id.get(t, 1) for t in tokens][:self.max_len]
        padding = [0] * (self.max_len - len(seq))
        return torch.tensor(seq + padding), torch.tensor(self.labels[idx])

In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return torch.sum(x * weights, dim=1), weights

In [ ]:
class EmotionalBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.attention = Attention(hidden_dim)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        lstm_out, _ = self.lstm(x)
        context, weights = self.attention(lstm_out)
        return self.fc(context)

In [ ]:
class HybridEliza:
    def __init__(self, lang="my", model_path="eliza_my.pth"):
        self.lang = lang
        #self.script = SCRIPTS[lang]
        #self.script["keywords"].sort(key=lambda x: x[2], reverse=True)

        self.model_path = model_path
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.word2id = {"<PAD>": 0, "<UNK>": 1}
        # Emotion labels in Myanmar (Burmese)
        self.id2label = {
            0: "Sadness", 1: "Joy", 2: "Love",
            3: "Anger", 4: "Fear", 5: "Surprise"
        }
        self.model = None


    def build_vocab(self, texts, vocab_size=8000):
        if self.lang == "my":
            all_tokens = [
                syl
                for t in texts
                for syl in myanmar_syllable_tokenize(normalize_myanmar(str(t)))
            ]
        else:
            all_tokens = [w for t in texts for w in str(t).lower().split()]

        counts = Counter(all_tokens)
        for i, (tok, _) in enumerate(counts.most_common(vocab_size), 2):
            self.word2id[tok] = i


    # Training
    def train(self, data_path, epochs, lr, batch_size, val_split=0.1):
        df = pd.read_csv(data_path).sample(frac=1, random_state=42).reset_index(drop=True)


        self.build_vocab(df['Text '])
        label_col = 'Label ' if 'Label ' in df.columns else 'emotions'

        full_dataset = EmotionDataset(
            df['Text '].tolist(), df[label_col].tolist(),
            self.word2id, max_len=100, lang=self.lang
        )
        val_size = int(len(full_dataset) * val_split)
        train_size = len(full_dataset) - val_size
        train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=batch_size)

        self.model = EmotionalBiLSTM(len(self.word2id), 128, 64, 6).to(self.device)
        optimizer  = optim.Adam(self.model.parameters(), lr=lr)
        criterion  = nn.CrossEntropyLoss()

        print(f"[*] Training on {self.device} | lang={self.lang} | vocab={len(self.word2id)}")
        for epoch in range(epochs):
            self.model.train()
            total_loss = 0
            for batch_x, batch_y in train_loader:
                batch_x, batch_y = batch_x.to(self.device), batch_y.to(self.device)
                optimizer.zero_grad()
                loss = criterion(self.model(batch_x), batch_y)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            val_acc = self.evaluate(val_loader)
            print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2%}")

        torch.save({'state': self.model.state_dict(), 'vocab': self.word2id}, self.model_path)
        print(f"[*] Model saved to {self.model_path}")

    def evaluate(self, loader):
        self.model.eval()
        correct = total = 0
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(self.device), y.to(self.device)
                _, predicted = torch.max(self.model(x), 1)
                total   += y.size(0)
                correct += (predicted == y).sum().item()
        return correct / total

    def load_model(self):
        if os.path.exists(self.model_path):
            ckpt = torch.load(self.model_path, map_location=self.device)
            self.word2id = ckpt['vocab']
            self.model = EmotionalBiLSTM(len(self.word2id), 128, 64, 6).to(self.device)
            self.model.load_state_dict(ckpt['state'])
            self.model.eval()
            print(f"[*] Model loaded from {self.model_path} | vocab={len(self.word2id)}")
        else:
            print(f"[!] No model found at {self.model_path}. Run with --mode train first.")

    def get_eq(self, text):
        if not self.model:
            return "Neutral", 0.0  # "Neutral" in Myanmar

        text = normalize_myanmar(text)
        # Strip Myanmar punctuation (analogous to English [^\w\s] strip)
        text_clean = normalize_myanmar(text)

        if self.lang == "my":
            tokens = myanmar_syllable_tokenize(text_clean)
        else:
            tokens = text_clean.lower().split()

        ids = [self.word2id.get(t, 1) for t in tokens][:100]
        ids += [0] * (100 - len(ids))

        with torch.no_grad():
            output = self.model(torch.tensor([ids]).to(self.device))
            probs  = torch.softmax(output, dim=1)
            idx    = torch.argmax(probs).item()
            return self.id2label[idx], probs[0][idx].item()

In [ ]:
data_path  = "/content/drive/MyDrive/burmese_emotion.csv"
lang       = "my"
epochs     = 20
batch_size = 32
lr         = 0.001
val_split  = 0.1
model_path = "eliza_my.pth"

In [ ]:
eliza = HybridEliza(lang=lang, model_path=model_path)
eliza.train(data_path, epochs, lr, batch_size, val_split)

[*] Training on cuda | lang=my | vocab=1270
Epoch 1/20 | Loss: 1.7750 | Val Acc: 35.83%
Epoch 2/20 | Loss: 1.5779 | Val Acc: 49.17%
Epoch 3/20 | Loss: 1.2299 | Val Acc: 77.50%
Epoch 4/20 | Loss: 0.7984 | Val Acc: 82.50%
Epoch 5/20 | Loss: 0.4950 | Val Acc: 82.50%
Epoch 6/20 | Loss: 0.2876 | Val Acc: 84.17%
Epoch 7/20 | Loss: 0.1714 | Val Acc: 88.33%
Epoch 8/20 | Loss: 0.1007 | Val Acc: 84.17%
Epoch 9/20 | Loss: 0.0542 | Val Acc: 85.83%
Epoch 10/20 | Loss: 0.0327 | Val Acc: 85.83%
Epoch 11/20 | Loss: 0.0208 | Val Acc: 85.83%
Epoch 12/20 | Loss: 0.0144 | Val Acc: 85.00%
Epoch 13/20 | Loss: 0.0110 | Val Acc: 85.83%
Epoch 14/20 | Loss: 0.0087 | Val Acc: 86.67%
Epoch 15/20 | Loss: 0.0071 | Val Acc: 85.00%
Epoch 16/20 | Loss: 0.0060 | Val Acc: 85.83%
Epoch 17/20 | Loss: 0.0051 | Val Acc: 85.00%
Epoch 18/20 | Loss: 0.0044 | Val Acc: 85.83%
Epoch 19/20 | Loss: 0.0038 | Val Acc: 85.00%
Epoch 20/20 | Loss: 0.0034 | Val Acc: 86.67%
[*] Model saved to eliza_my.pth


In [ ]:
user_in = "စောက်ရှက်ကို တစက်မှ မရှိတာဟေ့"
emotion, score = eliza.get_eq(user_in)
print(f"Emotion {emotion} ({score:.2%})")

Emotion Anger (99.93%)
